# 66 — Hallucination Testing
**Goal:** Detect and quantify LLM hallucinations in resume output.

Chapters 64–65 measured errors of *selection*: extracting or classifying the wrong thing. LLMs introduce a worse failure mode — errors of *invention*, where the model emits plausible content that is not in the source resume at all. This chapter defines the hallucination taxonomy for resume tasks and implements two automated detectors: a **grounding check** (is the output supported by the source text?) and a **consistency check** (are facts stable across rephrasings?).

**Why it matters for resumes / ATS:** a resume an LLM "improves" by inventing TensorFlow experience, a 5-year tenure, or a 40% metric is not just wrong — it misrepresents the candidate to employers and is a legal and reputational liability for any service that produces it. Automated hallucination checks are the guardrail that lets you keep the LLM rewrite while discarding the fabrication. In a pipeline that feeds extracted facts into matching or screening, an ungrounded fact is worse than a missed one: it actively lies about the candidate.

## 1. Types of Hallucination

Hallucinations are not one phenomenon — they come in distinct flavors with distinct costs. The code lists the four that matter for resumes: **skill hallucination** (claiming a skill absent from the resume), **experience hallucination** (inventing job details or titles), **date hallucination** (wrong or fabricated tenure), and **metric hallucination** (fabricated numbers such as "40% improvement"). The last two are the most dangerous because a reader cannot spot them by eye.

**What the code does:** prints the taxonomy alongside three detection strategies. A **grounding check** asks whether output content exists in the source text; a **consistency check** re-runs the task on paraphrases and compares facts across calls; a **factual check** validates against external knowledge (company locations, standard job titles). The three form a ladder: grounding is cheap and local, consistency catches stable-but-wrong inventions, and factual checks catch errors neither can see.

**Try it:** classify a few LLM outputs into the four types before running the detectors — the type shapes which detector can catch it.

In [ ]:
print('''Hallucination types in resume LLM tasks:
1. Skill hallucination — claiming a skill not in the resume
2. Experience hallucination — inventing job details
3. Date hallucination — wrong or made-up dates
4. Metric hallucination — fabricated numbers (40% improvement)

Detection strategies:
- Grounding check: Does output exist in source text?
- Consistency check: Same facts across multiple calls?
- Factual check: Known facts (e.g., company locations)''')

## 2. Grounding Checker

The grounding check implements the simplest faithful test: extract candidate claims from the model output and verify each appears verbatim in the source text. This version extracts claims with a regex for capitalized phrases (proper nouns and title-case spans), lowercases both sides for a case-insensitive match, and reports `grounded_pct` — the share of detected claims supported by the source. 100% means every claim the extractor found is backed by the resume.

**What the code does:** given the source "Python developer with NLP experience at Google" and an output that adds "TensorFlow" and "(5 years)", the regex `\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b` finds only `Python` and `Google` — both grounded, so it reports 100%. The catch is instructive: `TensorFlow` (mixed-case) and `NLP` (all-caps acronym) never match the `[A-Z][a-z]+` pattern, so the fabricated skill and the fabricated tenure are *invisible to the claim extractor*. A grounding check is only as strong as its claim splitter.

**Try it:** extend the pattern (e.g. allow interior capitals or acronym runs) and `TensorFlow` reappears — now flagged as ungrounded. The detector's blind spots are part of the result.

In [ ]:
import re

def grounding_check(output, source_text):
    """Check if output facts are grounded in source text."""
    # Tokenize into claims
    claims = re.findall(r"\\b[A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*\\b", output)
    source_lower = source_text.lower()
    
    grounded = []
    ungrounded = []
    for claim in claims:
        if len(claim) < 3: continue
        if claim.lower() in source_lower:
            grounded.append(claim)
        else:
            ungrounded.append(claim)
    
    return {
        "grounded": grounded[:5],
        "ungrounded": ungrounded[:5],
        "grounded_pct": len(grounded) / max(len(grounded + ungrounded), 1),
    }

source = "Python developer with NLP experience at Google"
output = "Python developer with NLP and TensorFlow experience at Google (5 years)"
result = grounding_check(output, source)
print(f"Grounded: {result['grounded_pct']:.0%}")
print(f"Grounded facts: {result['grounded']}")
print(f"Ungrounded: {result['ungrounded']} (possible hallucinations)")

## 3. Consistency Testing

A model that hallucinates the *same* way every time can defeat keyword-overlap grounding checks. Consistency testing exploits a different property of invention: it is unstable under paraphrase. Run the task several times with rephrased inputs; if the extracted facts stay identical across runs they are more likely true, and if facts appear, vanish, or change, that variation is the fingerprint of hallucination.

**What the code does:** extracts the skill set from the original text with a skill-vocabulary regex, then compares each paraphrase's skill set for equality. On the given example it reports consistency 67% (2 of 3): the first two variants preserve `{python, nlp, tensorflow}`, while the third — "Java developer with Spring Boot" — swaps in `{java}`. One inconsistent run in three is a loud signal that the model does not reliably preserve the resume's skill facts under reformulation.

**Try it:** note the equality test is strict — casing and order are normalized, but any added or dropped skill fails. For skills, "close" is not good enough.

In [ ]:
def consistency_check(text, variations):
    """Check if same facts are preserved across variations."""
    # Extract key facts from original
    skills = re.findall(r"\\b(Python|Java|NLP|TensorFlow|AWS|Docker)\\b", text, re.IGNORECASE)
    skills = set(s.lower() for s in skills)
    
    consistent = 0
    total = 0
    for var in variations:
        var_skills = set(s.lower() for s in re.findall(r"\\b(Python|Java|NLP|TensorFlow|AWS|Docker)\\b", var, re.IGNORECASE))
        if skills == var_skills:
            consistent += 1
        total += 1
    
    return {
        "original_skills": list(skills),
        "consistent_runs": consistent,
        "total_runs": total,
        "consistency": consistent / max(total, 1),
    }

original = "Python and NLP expert with TensorFlow"
variations = [
    "Python NLP specialist, TensorFlow",
    "Expert in Python and TensorFlow with NLP focus",
    "Java developer with Spring Boot",  # Wrong!
]
result = consistency_check(original, variations)
print(f"Consistency: {result['consistency']:.0%} ({result['consistent_runs']}/{result['total_runs']})")
print(f"Variant 3 was inconsistent (Java vs Python/TensorFlow/NLP)")

## Summary: Hallucination testing catches LLM fabrications. Always ground-check outputs against source text.

**Faithfulness is a measurable property — measure it, or ship fabrication.**

Grounding and consistency checks are the two cheapest automated guards in the hallucination toolbox: one verifies output against the source, the other verifies stability across paraphrases, and each catches failure modes the other cannot see. The worked examples also expose the tooling's blind spots — regex claim extraction misses mixed-case and acronym tokens — so treat these scores as lower bounds on suspicion, not certificates of truth. These detectors are the evaluation counterpart to the LLM rewriting steps of earlier blocks: they let you keep the rewrite while discarding the invention. The next chapter turns to optimizing prompts themselves, with A/B testing against a golden set.